# §8 体验实验：LoRA 加一对小矩阵

这个笔记本只做一件事：把 LoRA 的全部算术拆开给你亲手算一遍——**秩是什么**（d=4、r=2 的手算链，从 B、A 一直算到输出 h）、**参数量账**（r 取 1/2/4/8 时 2dr 对 d² 的倍数账）、再用一段几十行的 toy 代码，看**秩 r 从小调到大时误差怎么降、为什么降过头反而浪费**（低秩饱和）。最后用三张纸面账回答：QLoRA 凭什么让 24GB 的消费级显卡微调 7B 大模型。

- **一条主线**：W = W₀ + BA。W₀ 冻结不动，只训一对小矩阵 B（d×r，高瘦）和 A（r×d，矮胖），BA 的秩最多只有 r。
- **为什么值得手算**：LoRA 是高效微调的当家方法，QLoRA 又在它身上叠了 4-bit 量化。这一页纸的矩阵乘法加三笔账，就是它全部的心脏。
- **安装说明**：Colab 已预装 NumPy，直接运行即可，无需 pip 单元格；全程不画图，数值表就是结果。

本笔记本中的代码与输出均来自 2026-10-09 的真实运行记录。

## 1. 秩是什么：d=4、r=2 的 BA 手算链

秩，是矩阵真正能“撑开”的**独立方向的数量**。LoRA 里这对小矩阵的成绩单就写在秩上：BA 的秩最多等于 r——B 只有 r 列，乘出来的矩阵再怎么努力也挤不出第 r+1 个独立方向。

✏️ **拿张纸，先别运行下面的格子**：照正文同款的数字手算一遍——

$$W_0 = I_4,\quad B = \begin{bmatrix} 1&2\\3&4\\5&6\\7&8\end{bmatrix},\quad A = \begin{bmatrix} 1&0&1&0\\0&1&0&1\end{bmatrix},\quad x = [1,2,3,4]^T$$

第一步算 BA（4×2 乘 2×4，得 4×4），观察它的第 1 列和第 3 列——发现了什么？第二步算 W = W₀ + BA，第三步算 h = Wx。三行数字写下来再往下走。

In [ ]:
import numpy as np

# 题面照对账记录 4a（旧版 L240–256），与正文手算例逐字同源
W0 = np.eye(4)                                    # 预训练权重：这里取 I₄，冻结不动
B = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])    # 高瘦：4×2
A = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])        # 矮胖：2×4

BA = B @ A
print('BA ='); print(BA)
print('第 1 列 == 第 3 列、第 2 列 == 第 4 列：独立方向只有 2 个 → 秩恰好等于 r = 2，'
      f'np 复核 rank(BA) = {np.linalg.matrix_rank(BA)}')

W = W0 + BA
x = np.array([1, 2, 3, 4])
h = W @ x
print(f'\nW = W₀ + BA ='); print(W.astype(int))
print(f'h = Wx = {h.tolist()}')
print(f'拆账：底子 W₀x = {(W0 @ x).tolist()}（预训练），调料 BAx = {(BA @ x).tolist()}（LoRA 修正）')

## 2. 参数量账填空：r 越小，省的越多还是越少？

对 d=64 的方阵，全量微调要动 d²=4096 个参数；LoRA 只训一对 B、A，共 **2dr** 个。✏️ **把下表四行三列填完再运行**（最后一列提示：压缩倍数 = d/2r）：

| r | LoRA 参数 2dr | 原矩阵 d² | 压缩倍数 |
|---|---|---|---|
| 1 |  | 4096 |  |
| 2 |  | 4096 |  |
| 4 |  | 4096 |  |
| 8 |  | 4096 |  |

填的时候想一想：倍数账是 r 越大多省还是 r 越小多省？那 r 还有必要调大吗——先记住你的直觉，第 3、4 节用数据回答。

In [ ]:
d = 64
print(f'{"r":<4}{"LoRA 2dr":>10}{"原矩阵 d²":>12}{"压缩倍数 d/2r":>16}')
print('-' * 42)
for r in [1, 2, 4, 8]:
    print(f'{r:<4}{2*d*r:>10}{d*d:>12}{d//(2*r):>12} 倍')
print('-' * 42)
# 答案：r=1 → 128 / 32 倍；r=2 → 256 / 16 倍；r=4 → 512 / 8 倍；r=8 → 1024 / 4 倍
# r 越小账面上越省，但省的不是白省的——r 同时是修正能力的上限（下一节看代价）

## 3. toy LoRA：真秩 4 的合成矩阵，r=1…16 逐档拟合

现在把 d 放大到 64，造一个**真秩恰好为 4** 的目标矩阵 W\* = B\*A\*（B\* 是 64×4，A\* 是 4×64），再掺一点噪声当“真实观测”。对每个 r，找秩不超过 r 的矩阵里**最小二乘最优**的那个——Eckart–Young 定理保证它就是截断 SVD：把 Ŵ 的前 r 个奇异值留下、其余扔掉。

做法分两步，都是最小二乘：先用最小二乘从观测里估计全矩阵 Ŵ，再对每个 r 截断出最优 BA。打印两张误差表：拟合目标 ‖Ŵ−BA‖\_F，和对真值 ‖W\*−BA‖\_F。后者才是“真学到了多少”的账。

In [ ]:
rng = np.random.default_rng(8)             # 固定种子，结果可复现
d, R_TRUE, N = 64, 4, 256                  # 维度 / 真秩 / 线性观测条数

B_star = rng.normal(size=(d, R_TRUE))      # 高瘦：64×4
A_star = rng.normal(size=(R_TRUE, d))      # 矮胖：4×64
W_star = B_star @ A_star                   # 合成真值：64×64，秩恰为 4
X = rng.normal(size=(N, d))                # 线性观测输入
Y = X @ W_star + 0.05 * rng.normal(size=(N, d))   # 观测含少量噪声

W_hat = np.linalg.lstsq(X, Y, rcond=None)[0]        # 第一步：最小二乘估全矩阵
U, s, Vt = np.linalg.svd(W_hat)                     # 第二步：每个 r 取截断 SVD
print(f'd={d}, 真秩={R_TRUE}, 观测={N} 条（含少量噪声）')
print(f'{"r":>3} {"‖Ŵ−BA‖F(拟合目标)":>22} {"‖W*−BA‖F(真值误差)":>22}')
print('-' * 49)
obj, true_err = [], []
for r in range(1, 17):
    BA_r = (U[:, :r] * s[:r]) @ Vt[:r]     # rank ≤ r 的最优低秩近似
    obj.append(np.linalg.norm(W_hat - BA_r, 'fro'))
    true_err.append(np.linalg.norm(W_star - BA_r, 'fro'))
    print(f'{r:>3} {obj[-1]:>22.4f} {true_err[-1]:>22.4f}')
print('-' * 49)

## 4. 断言格：误差单调、r=4 后饱和（低秩饱和现象）

看第 3 节两张表的三件事：

1. **拟合目标随 r 单调不增**——秩越大表达能力越强，这是理所当然的；
2. **真值误差在 r=4（=真秩）处触底**：r<4 是欠拟合，欠着的那几个方向装不下真信息；
3. **r>4 后真值误差不降反微升**——拟合目标还在降，但降的那部分是噪声。这就是**低秩饱和**：过了真秩，参数翻倍换来的只是对噪声的记忆力。

这直接回答第 2 节的问题：r 越小账面越省，但 r=1 的欠拟合误差是 r=4 的一千多倍——**省的不是白省的，r 是效率与表达力的权衡，不是越小越好**。下面让计算机当裁判。

In [ ]:
# 断言一：拟合误差随 r 单调不增（容差 1e-8，应对浮点抖动）
assert all(obj[i + 1] <= obj[i] + 1e-8 for i in range(15)), '拟合误差应随 r 单调不增'
# 断言二：低秩饱和——r=真秩 4 时真值误差最小；r>4 误差不降反微升（拟合到噪声）
assert true_err[15] > true_err[3], 'r>真秩后应出现过拟合：真值误差回升'
assert true_err[3] < 0.02 * obj[0], 'r=真秩时误差应降到 r=1 欠拟合的 2% 以内'
print(f'对账：误差单调不增 ✓；r=4 真值误差 {true_err[3]:.3f} 触底、r=16 回升到 {true_err[15]:.3f} ✓')
print(f'     r=1 欠拟合 {obj[0]:.3f}，为 r=4 误差的 {obj[0] / true_err[3]:.0f} 倍')
print('对账：低秩饱和现象复现，r 过小不够、过大浪费 → PASS')

## 5. QLoRA 三账题卡：4-bit 显存怎么 ÷4

最后一站是纸面账（QLoRA 完整训练要 7B 权重与多小时算力，超出本章能给的动手范围；链路本身可小规模真跑，见末节与 run_log；结论方向有论文核实，见同目录《QLoRA引用卡》）。

QLoRA = 4-bit 量化的 W₀（冻结）+ 16-bit 的 LoRA（B、A）。✏️ **填下面三笔账再运行**（7B 参数、十进制口径，即 1GB = 10⁹ 字节）：

**账一（位宽账）**：32 位变 4 位，存储量是原来的几分之几？

**账二（显存三档）**：7B 参数分别用 32-bit / 16-bit / 4-bit 存放，各要多少 GB？（提示：参数量 × 每参数字节数。）

**账三（能不能装下）**：24GB 显存的消费级显卡（如 RTX 3090/4090），三档各装得下吗？哪一档让“消费级显卡微调 7B”从不可能变成宽裕？再想想 13B 用 4-bit 要多少 GB？

In [ ]:
p7b = 7e9   # 7B 参数（十进制：7×10⁹）
g32 = p7b * 4 / 1e9    # 账二·32-bit：7×4 字节 = 28 GB
g16 = p7b * 2 / 1e9    # 账二·16-bit：7×2 字节 = 14 GB
g4 = p7b * 0.5 / 1e9   # 账二·4-bit：14 ÷ 4 = 3.5 GB（相对 32-bit 是 28 ÷ 8）

print('账一：32→4 位，存储量 = 4/32 = 1/8（位宽比；相对 16-bit 是 16/4 = ÷4 的账）')
print(f'账二：32-bit ≈ {g32:.1f} GB，16-bit ≈ {g16:.1f} GB，4-bit ≈ {g4:.1f} GB')
print(f'账三：28 GB 装不下 24GB 卡；14 GB 紧张（还没算优化器状态）；'
      f'{g4:.1f} GB 宽裕 → 4-bit 让消费级卡微调 7B 成立；13B×0.5 字节 ≈ {13e9*0.5/1e9:.1f} GB 也装得下')

# 断言格：三笔账与对账记录 4a（旧版 L263：16-bit≈14GB、32-bit≈28GB；L267：32→4 位 = 1/8）逐格一致
assert abs(g32 - 28) < 1e-9 and abs(g16 - 14) < 1e-9 and abs(g4 - 3.5) < 1e-9
assert abs(28 / 8 - 3.5) < 1e-9, '账一：4-bit 应是 32-bit 的 1/8'
print('对账：位宽账 + 显存三档（28/14/3.5 GB）与对账记录 4a 逐格一致 → PASS')

## 6. 口径与边界：三个容易说错的地方

- **“推理零额外开销”说的是合并之后**：训练完把 BA 直接加回 W₀，推理时的矩阵和原来一样大——零额外延迟、零额外显存。但训练时前向仍要算 BA，不是零开销。LoRA 省的是**训练**成本（参数、优化器状态、反向计算）。
- **显存账统一十进制**：7B×2 字节=14GB、×4 字节=28GB 都是 1GB=10⁹ 字节的口径；若用二进制 GiB 会得到 13.0/26.1，两套口径不能混排。“4-bit=1/8”是 32→4 的位宽比，别写成“比 16-bit 省一半”——相对 16-bit 是 ÷4。
- **“r 小不一定差”是有条件的**：第 4 节的 toy 里 r=1 远不够；QLoRA 论文的消融显示，当 LoRA 挂在**全部线性层**上时，r 的影响远小于“挂在哪几层”——此时较小的 r 也能打平。真实任务里 r 的取舍请带数据说话，论文方向性结论与逐档数字见《QLoRA引用卡》。

In [ ]:
# 与正文参数量主表对账（对账记录 4a，旧版 L222–226）：d² vs 2dr，压缩倍数 = d/2r
rows = [(512, 4, 262144, 4096, 64), (4096, 4, 16777216, 32768, 512),
        (12288, 4, 150994944, 98304, 1536)]
print(f'{"d":<8}{"r":<4}{"全量 d²":>14}{"LoRA 2dr":>12}{"压缩倍数":>10}')
print('-' * 50)
for d, r, full, lora, ratio in rows:
    assert d * d == full and 2 * d * r == lora and d // (2 * r) == ratio
    print(f'{d:<8}{r:<4}{full:>14,}{lora:>12,}{ratio:>8} 倍')
print('-' * 50)
print('对账：正文三行表（64/512/1,536 倍，含“1600万→3.3万”的 512 倍口播版）'
      '与对账记录 4a 逐格一致 → PASS')

## 7. 要点回顾

- **秩是独立方向的数量**：BA 的秩 ≤ r，d=4、r=2 的手算链里 BA 第 1 列=第 3 列，秩恰好是 2；W = W₀ + BA 把预训练底子和 LoRA 调料拆得清清楚楚（底子 [1,2,3,4]ᵀ + 调料 [16,36,56,76]ᵀ = h）。
- **参数量账：2dr 对 d²，压缩倍数 = d/2r**。d=64 时 r=1/2/4/8 对应 32/16/8/4 倍；d=4096、r=4 是正文那笔 512 倍的账（1,600 万 → 3.3 万）。
- **r 是权衡不是越小越好**：toy 实验里 r=1 的欠拟合误差是 r=4 的一千多倍，r>4 后真值误差不降反升——低秩饱和。真实任务里“r 小也能打平”依赖 LoRA 挂全部线性层等条件（QLoRA 论文，见引用卡）。
- **QLoRA 的三笔账**：32→4 位存储变 1/8；7B 显存三档 28/14/3.5 GB；4-bit 让 24GB 消费级卡微调 7B（乃至 13B）成立。省显存靠量化，省训练靠 LoRA，两者互补。
- 实测打印原文、机器环境与判定结果见同目录《run_log.txt》；QLoRA 论文核实结论见《QLoRA引用卡》。

## 8. 链路可跑验证：NF4 底座 + LoRA（小规模）

结业项目二的第三件：把 QLoRA 链路真跑一遍——冻结的 4-bit 底座、16 位适配器、十几步训练、adapter 存取、推理 smoke。完整收敛（两组秩对比、多小时）仍留给租卡；这一格验证的是**链路能跑**（本机 CPU 实测：加载 41s、单步 6.5s、峰值内存 11.6GB、12 步 loss 3.21→0.52、adapter 19.3MB）。

> 前置：约 15GB 权重下载一次；依赖 `transformers peft bitsandbytes`。脚本与运行记录见同目录 `qlora_7b_cpu_smoke.py`、`run_log_qlora_7b_cpu_smoke.txt`。

In [ ]:
# 链路五断言：NF4 加载 / 可训练参数 / loss 下降 / adapter 存盘 / 推理 smoke
# 时间盒 900s、12 步；完整输出见同目录 run_log_qlora_7b_cpu_smoke.txt
!python qlora_7b_cpu_smoke.py --steps 12 --box 900